# Упражнение: TCN: каузальная дилатированная свёртка и рецептивное поле

## Задание

В `input.csv` — ряд из 24 точек (`t, x`), в `weights.json` — веса трёх одноканальных свёрточных слоёв
с ядром k = 3: `w1, b1`, `w2, b2`, `w3, b3` (веса — списки из 3 чисел, смещения — числа).

Каузальная дилатированная свёртка (формула из лекции; ряд слева дополняется (k − 1)·d нулями):

y(t) = b + Σⱼ w[j] · x(t − j·d),  j = 0..2,  x(t′) = 0 при t′ < 0

То есть `w[0]` умножается на текущий момент, `w[1]` — на d шагов назад, `w[2]` — на 2d шагов назад.

Сеть (один остаточный блок в миниатюре):

1. h1 = ReLU(conv(x; w1, b1, d = 1))
2. h2 = ReLU(conv(h1; w2, b2, d = 2))
3. out = x + conv(h2; w3, b3, d = 4) — остаточная связь: вход прибавляется к выходу.

Также посчитайте рецептивное поле стека R = 1 + (k − 1)·(d₁ + d₂ + d₃) и для каждого t — число входных точек
`n_inputs`, от которых структурно зависит out(t): сколько моментов из окна [t − R + 1, t] реально существуют
(не уходят левее t = 0).

## Подсказки

- Каузальность проверяется просто: измените последнюю точку x — ни один выход в более ранние моменты
  не должен измениться.
- R = 1 + 2·(1 + 2 + 4) = 15; поэтому `n_inputs` растёт 1, 2, 3, … в начале ряда и дальше постоянен.
- `xp = np.concatenate([np.zeros((k-1)*d), x])`, затем для каждого t:
  `b + sum(w[j] * xp[pad + t - j*d] for j in range(3))`. Дополнение нулями делается на входе **каждого** слоя
  (для слоя 2 — нули перед h1, а не ReLU(b1)).

## Ожидаемый формат выходного файла

CSV с колонками `t, h1, h2, out, n_inputs` — 24 строки (t = 0..23 по порядку);
значения округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/cnn_time_series_tcn
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.